# Build 06 — Causal Inference Mitigation

> *DoWhy + IPW debiased retraining to break the SFP loop at training time*

**EN:** Final build. Specify a causal DAG, use the backdoor criterion to identify the confounding path, then retrain on IPW-weighted data to remove bias. Compare debiased vs standard models on AUC, recall, Brier, calibration error — overall and at segment level.

**KR:** 최종 빌드. 인과 DAG 명시, backdoor criterion으로 교란 경로 식별, IPW 가중 데이터로 재학습해 편향 제거. debiased vs standard 모델을 AUC·recall·Brier·calibration 기준으로 비교 — 전체 및 segment.

## 1. Imports (+ optional DoWhy)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

try:
    import dowhy
    from dowhy import CausalModel
    DOWHY_AVAILABLE = True
except ImportError:
    DOWHY_AVAILABLE = False
    print("DoWhy not installed. Running IPW pipeline without DoWhy. Install with: pip install dowhy")


FEATURE_COLS = ['high_amount', 'night_claim', 'high_postcode', 'prior_claims']

## 2. Causal DAG Specifications (GML + DOT)

In [ ]:
CAUSAL_DAG_GML = """
graph [
    directed 1
    node [ id "claim_features" label "claim_features" ]
    node [ id "true_risk" label "true_risk" ]
    node [ id "model_score" label "model_score" ]
    node [ id "investigation" label "investigation" ]
    node [ id "fraud_label" label "fraud_label" ]
    edge [ source "true_risk" target "fraud_label" ]
    edge [ source "true_risk" target "claim_features" ]
    edge [ source "claim_features" target "model_score" ]
    edge [ source "model_score" target "investigation" ]
    edge [ source "investigation" target "fraud_label" ]
]
"""

# DoWhy dot notation (alternative format)
CAUSAL_DAG_DOT = """
digraph {
    true_risk -> fraud_label;
    true_risk -> claim_features;
    claim_features -> model_score;
    model_score -> investigation;
    investigation -> fraud_label;
}
"""

## 3. DoWhy Causal Analysis (optional)

In [ ]:
def run_dowhy_analysis(
    df: pd.DataFrame,
    treatment_col: str = 'investigated',
    outcome_col: str = 'observed_fraud',
    feature_cols: list[str] = None,
) -> dict:
    """
    Use DoWhy to formally identify and estimate the causal effect of
    investigation on fraud labels.

    This separates:
        - True causal effect: investigation CAUSES fraud discovery (= loop signal)
        - Spurious correlation: high-risk claims are both more likely investigated
          AND more likely fraudulent (= legitimate model signal)
    """
    if not DOWHY_AVAILABLE:
        return {'error': 'DoWhy not available. Run: pip install dowhy'}

    if feature_cols is None:
        feature_cols = FEATURE_COLS

    # Only use investigated claims (where outcome is observed)
    invest_mask = df[treatment_col] == 1
    analysis_df = df[invest_mask].dropna(subset=[outcome_col]).copy()
    analysis_df[outcome_col] = analysis_df[outcome_col].astype(int)
    analysis_df[treatment_col] = analysis_df[treatment_col].astype(int)

    # Columns needed for DoWhy
    needed_cols = [treatment_col, outcome_col] + feature_cols
    analysis_df = analysis_df[needed_cols].dropna()

    try:
        model = CausalModel(
            data=analysis_df,
            treatment=treatment_col,
            outcome=outcome_col,
            graph=CAUSAL_DAG_DOT,
            common_causes=feature_cols,
        )

        identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Estimate using propensity score weighting
        estimate = model.estimate_effect(
            identified_estimand,
            method_name="backdoor.propensity_score_weighting",
            confidence_intervals=True,
        )

        # Refutation: random common cause test
        refute = model.refute_estimate(
            identified_estimand, estimate,
            method_name="random_common_cause",
            num_simulations=10,
        )

        return {
            'causal_estimate': round(estimate.value, 4),
            'confidence_interval': estimate.get_confidence_intervals(),
            'refutation_p_value': round(refute.new_effect, 4) if hasattr(refute, 'new_effect') else None,
            'interpretation': (
                f"Causal effect of investigation on fraud label: {estimate.value:.4f}. "
                f"This is the portion of the observed fraud rate attributable to the "
                f"investigation decision itself (loop signal), not underlying claim risk."
            )
        }

    except Exception as e:
        return {'error': f'DoWhy estimation failed: {str(e)}'}

## 4. IPW Debiased Training

In [ ]:
def train_ipw_debiased_model(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    investigation_col: str = 'investigated',
    fraud_col: str = 'observed_fraud',
    score_col: str = None,
    clip_weights: float = 10.0,
) -> tuple[LogisticRegression, np.ndarray]:
    """
    Train a logistic regression model on IPW-weighted training data.

    IPW weight for claim i:
        w_i = 1 / P(investigated=1 | features_i)

    This gives higher weight to claims that were investigated DESPITE
    having low model scores (random or near-threshold investigations),
    making the training set more representative of the full population.

    Args:
        clip_weights: Maximum allowed weight (prevents extreme influence)

    Returns:
        (debiased_model, ipw_weights)
    """
    # Estimate propensity scores
    X_ps = train_df[feature_cols].values
    y_ps = train_df[investigation_col].values.astype(int)
    ps_model = LogisticRegression(max_iter=1000)
    ps_model.fit(X_ps, y_ps)
    propensity = ps_model.predict_proba(X_ps)[:, 1]
    propensity = np.clip(propensity, 0.01, 0.99)

    # IPW weights
    ipw_weights = 1.0 / propensity
    ipw_weights = np.clip(ipw_weights, 1.0, clip_weights)  # clip extremes

    # Train debiased model on INVESTIGATED claims with IPW weights
    invest_mask = train_df[investigation_col] == 1
    fraud_labels = train_df.loc[invest_mask, fraud_col].dropna()
    invest_idx = train_df[invest_mask].index

    X_train = train_df.loc[invest_idx, feature_cols].values
    y_train = fraud_labels.values.astype(int)
    w_train = ipw_weights[invest_mask][:len(y_train)]

    if len(np.unique(y_train)) < 2:
        raise ValueError("Training labels have only one class after filtering — need more data")

    debiased_model = LogisticRegression(max_iter=1000, class_weight='balanced')
    debiased_model.fit(X_train, y_train, sample_weight=w_train)

    print(f"IPW debiased model trained on {len(y_train):,} investigated claims")
    print(f"  Mean weight: {w_train.mean():.3f}")
    print(f"  Max weight (clipped at {clip_weights}): {w_train.max():.3f}")
    print(f"  Weight Gini (dispersion): {np.std(w_train) / np.mean(w_train):.3f}")

    return debiased_model, ipw_weights

## 5. Standard (Biased) Training

In [ ]:
def train_standard_model(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    investigation_col: str = 'investigated',
    fraud_col: str = 'observed_fraud',
) -> LogisticRegression:
    """Train standard (biased) model on investigated claims without re-weighting."""
    invest_mask = train_df[investigation_col] == 1
    sub = train_df[invest_mask].dropna(subset=[fraud_col])

    X = sub[feature_cols].values
    y = sub[fraud_col].values.astype(int)

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X, y)
    return model

## 6. Evaluation vs Ground Truth

In [ ]:
def evaluate_models(
    test_df: pd.DataFrame,
    standard_model: LogisticRegression,
    debiased_model: LogisticRegression,
    feature_cols: list[str],
    true_fraud_col: str = 'true_fraud',
    threshold: float = 0.5,
) -> pd.DataFrame:
    """
    Compare debiased vs standard model against true fraud labels (ground truth).

    In production, true_fraud_col would not exist — we'd use IPS-corrected metrics.
    Here we use ground truth for clean validation.
    """
    X_test = test_df[feature_cols].values
    true_labels = test_df[true_fraud_col].values

    scores_standard = standard_model.predict_proba(X_test)[:, 1]
    scores_debiased = debiased_model.predict_proba(X_test)[:, 1]

    preds_standard = (scores_standard >= threshold).astype(int)
    preds_debiased = (scores_debiased >= threshold).astype(int)

    results = []
    for model_name, scores, preds in [
        ('Standard (biased)', scores_standard, preds_standard),
        ('IPW Debiased', scores_debiased, preds_debiased),
    ]:
        auc = roc_auc_score(true_labels, scores)
        recall = recall_score(true_labels, preds, zero_division=0)
        precision = (
            np.sum((preds == 1) & (true_labels == 1)) /
            max(np.sum(preds == 1), 1)
        )
        brier = brier_score_loss(true_labels, scores)

        # Calibration: compare predicted rate vs actual rate in deciles
        deciles = pd.qcut(scores, 10, duplicates='drop', labels=False)
        calib_df = pd.DataFrame({'score': scores, 'true_fraud': true_labels, 'decile': deciles})
        calib = calib_df.groupby('decile').agg(
            pred_rate=('score', 'mean'),
            actual_rate=('true_fraud', 'mean')
        )
        calibration_error = np.mean(np.abs(calib['pred_rate'] - calib['actual_rate']))

        results.append({
            'Model': model_name,
            'AUC (vs true fraud)': round(auc, 4),
            'Recall (vs true fraud)': round(recall, 4),
            'Precision (vs true fraud)': round(precision, 4),
            'Brier Score': round(brier, 4),
            'Calibration Error (ECE)': round(calibration_error, 4),
        })

    return pd.DataFrame(results)

## 7. Segment-Level Comparison

In [ ]:
def segment_level_comparison(
    test_df: pd.DataFrame,
    standard_model: LogisticRegression,
    debiased_model: LogisticRegression,
    feature_cols: list[str],
    segment_col: str,
    true_fraud_col: str = 'true_fraud',
) -> pd.DataFrame:
    """
    Compare models at segment level.

    The debiased model should show larger improvements in:
    - Segments that were historically under-investigated (blind spots)
    - High-postcode, night-time claims
    """
    X_test = test_df[feature_cols].values
    scores_standard = standard_model.predict_proba(X_test)[:, 1]
    scores_debiased = debiased_model.predict_proba(X_test)[:, 1]

    test_df = test_df.copy()
    test_df['score_standard'] = scores_standard
    test_df['score_debiased'] = scores_debiased

    agg = test_df.groupby(segment_col).apply(
        lambda g: pd.Series({
            'n_claims': len(g),
            'true_fraud_rate': g[true_fraud_col].mean(),
            'predicted_fraud_rate_standard': g['score_standard'].mean(),
            'predicted_fraud_rate_debiased': g['score_debiased'].mean(),
            'auc_standard': roc_auc_score(g[true_fraud_col], g['score_standard']) if g[true_fraud_col].nunique() > 1 else np.nan,
            'auc_debiased': roc_auc_score(g[true_fraud_col], g['score_debiased']) if g[true_fraud_col].nunique() > 1 else np.nan,
        })
    ).reset_index()

    agg['auc_improvement'] = agg['auc_debiased'] - agg['auc_standard']
    agg = agg.sort_values('auc_improvement', ascending=False)

    return agg

---

# 👁️ Section A — Hands-on: Train Two Models, Compare on Ground Truth

> *We generate looped data, train both standard and IPW debiased, evaluate on `true_fraud`.*

**EN:** This is the most important demo. The headline: standard AUC looks great, but IPW recovers recall on the loop's blind spots. Segment-level comparison shows where the debiased model wins.

**KR:** 가장 중요한 데모. 핵심: 표준 AUC는 좋아 보이지만 IPW는 루프의 blind spot에서 recall 회복. segment 단위 비교로 debiased가 어디서 이기는지 확인.

## A.1 Generate data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent / '01_sfp_simulation'))
from simulate_sfp_loop import run_full_simulation

print("Generating 20k claims with active SFP loop (ε = 0)...")
df, _ = run_full_simulation(n_claims=20_000, n_versions=3, epsilon=0.0, seed=42)

df['investigated'] = df['model_v3_investigated'].astype(int)
df['observed_fraud'] = pd.to_numeric(df['model_v3_observed_fraud'], errors='coerce')

# Temporal-style train/test split (first 70% / last 30%)
cutoff = int(len(df) * 0.7)
train_df = df.iloc[:cutoff].copy()
test_df  = df.iloc[cutoff:].copy()

print(f"\n📊 Train: {len(train_df):,}  ({train_df['investigated'].sum():,} investigated, "
      f"{train_df['investigated'].mean():.1%})")
print(f"📊 Test : {len(test_df):,}  ({test_df['investigated'].sum():,} investigated)")
print(f"   True fraud rate (train): {train_df['true_fraud'].mean():.3f}")
print(f"   True fraud rate (test) : {test_df['true_fraud'].mean():.3f}")

## A.2 Train standard (biased) model

In [ ]:
print("Training standard model on investigated claims only (no reweighting)...")
standard_model = train_standard_model(train_df, FEATURE_COLS, 'investigated', 'observed_fraud')
print(f"✅ Trained. Coefficients: {dict(zip(FEATURE_COLS, standard_model.coef_[0].round(3)))}")

## A.3 Train IPW debiased model

**EN:** Note the printed weight statistics — high max weight indicates strong selection bias and unstable IPW. Clipping is essential.

**KR:** 출력된 가중치 통계를 보세요 — max weight가 높으면 강한 selection bias와 IPW 불안정. 클리핑 필수.

In [ ]:
print("Training IPW debiased model (sample_weight = 1/propensity, clipped at 10)...")
debiased_model, ipw_weights = train_ipw_debiased_model(
    train_df, FEATURE_COLS, 'investigated', 'observed_fraud', clip_weights=10.0
)
print(f"✅ Trained. Coefficients: {dict(zip(FEATURE_COLS, debiased_model.coef_[0].round(3)))}")

## A.4 Evaluation against ground truth

**EN:** Both models are evaluated on `true_fraud`. The debiased model typically shows slightly lower AUC but higher recall, lower Brier, lower calibration error.

**KR:** 두 모델 모두 `true_fraud`로 평가. debiased 모델은 보통 AUC는 약간 낮지만 recall ↑, Brier ↓, calibration error ↓.

In [ ]:
eval_df = evaluate_models(
    test_df=test_df,
    standard_model=standard_model,
    debiased_model=debiased_model,
    feature_cols=FEATURE_COLS,
    true_fraud_col='true_fraud',
)
print("\n📊 EVALUATION vs GROUND TRUTH")
print("─" * 80)
print(eval_df.to_string(index=False))

## A.5 Segment-level: where does debiased win?

**EN:** Compare AUC per segment. The biggest gains should be in historically under-investigated areas (high postcode risk, night claims) — that's the evidence that IPW restored signal where the loop suppressed it.

**KR:** segment별 AUC 비교. 가장 큰 이득은 역사적으로 조사가 적었던 영역(고위험 우편번호, 야간 클레임) — IPW가 루프로 억압된 신호를 복원했다는 증거.

In [ ]:
for seg in ['high_postcode', 'night_claim']:
    print(f"\n📊 Segment: {seg}")
    seg_df = segment_level_comparison(
        test_df=test_df,
        standard_model=standard_model,
        debiased_model=debiased_model,
        feature_cols=FEATURE_COLS,
        segment_col=seg,
        true_fraud_col='true_fraud',
    )
    print(seg_df[[seg, 'n_claims', 'true_fraud_rate',
                  'auc_standard', 'auc_debiased', 'auc_improvement']].to_string(index=False))

## A.6 (Optional) DoWhy formal causal estimate

In [ ]:
if DOWHY_AVAILABLE:
    result = run_dowhy_analysis(
        train_df, 'investigated', 'observed_fraud', FEATURE_COLS
    )
    if 'error' not in result:
        print("📊 DoWhy estimate:")
        for k, v in result.items():
            if k != 'interpretation':
                print(f"  {k}: {v}")
        print(f"\n💡 {result['interpretation']}")
    else:
        print(f"⚠️  {result['error']}")
else:
    print("DoWhy not installed — skip with: pip install dowhy")

## 🎯 What you should observe

**EN:**
- **Standard AUC** typically higher than IPW debiased — but this is *biased* against the same selection
- **IPW debiased**: lower AUC, but *higher recall against true fraud* and lower calibration error
- **Segment level**: IPW wins where standard model neglected — high postcode risk, night claims
- The trade-off is real: IPW sacrifices some overall ranking quality to recover coverage in blind spots

**KR:**
- **표준 AUC**가 보통 IPW debiased보다 높음 — 하지만 이건 동일 selection에 대해 **편향**된 비교
- **IPW debiased**: AUC는 낮지만 **true fraud 기준 recall은 높음**, calibration error도 낮음
- **Segment 단위**: 표준 모델이 무시했던 곳에서 IPW 승 — 고위험 우편번호, 야간 클레임
- 트레이드오프 실재: IPW는 전체 순위 품질을 일부 희생하고 blind spot의 coverage를 회복